# Grid & Tile Alignment
Visualises the global 10km grid and sensor output tiles on an interactive map, and verifies their spatial alignment.

### Tiling method change log

**The problem we had before:**
Each MajorTOM grid cell is defined in UTM coordinates (metres). But we were telling SNAP which area to cut using a lon/lat polygon.
SNAP had to convert that lon/lat polygon back into pixel row/column numbers itself, and its conversion was slightly imprecise — tiles ended up shifted by a few pixels from the true grid cell boundary.
On top of that, the corner coordinates written into the `.h5` metadata were never updated after cutting, so they still described the original full-scene acquisition footprint, not the actual tile.

**What we do now:**
1. Compute the exact UTM bounding box of the grid cell (in metres).
2. Read the raster's pixel-to-UTM transform from the BEAM-DIMAP `.dim` file.
3. Convert the UTM bbox to exact pixel coordinates ourselves (`_utm_bbox_to_pixel_region`).
4. Tell SNAP to cut using those pixel coordinates directly (`region=x,y,width,height`) — no coordinate conversion inside SNAP.
5. After cutting, overwrite the corner coordinates in the `.h5` metadata with the correct WGS84 corners of the tile (`_update_h5_corners`).

**Result:** every tile is cut to exactly the right pixel boundary, and the metadata correctly describes the tile's actual extent.
This notebook verifies both properties.

> **Note:** this applies to Sentinel-1 only. NISAR tiles are cut directly in Python (`NISARCutter.cut_by_bbox`) without SNAP, so steps 2–5 do not apply — but the same UTM bbox logic (step 1) is used.

In [1]:
# jupyter trust /shared/home/egm/Projects/WorldSAR/pretraining/WORLDSAR/notebooks/visualise_grid_tiles.ipynb

import json
import h5py
import numpy as np
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Point, Polygon
import pyproj
from sarpyx.utils.viz import show_image
from sarpyx.utils.geos import GridNavigator

GRID_PATH = Path("../grid/grid_10km.geojson")
TILES_DIR = Path("/shared/home/egm/Projects/WorldSAR/pretraining/WORLDSAR/outputs/tiles")

# S1 TOPS swaths to include
S1_SWATHS = ("IW1",)

# NISAR product dirs: any subdir of TILES_DIR not matching a known swath name
_SWATH_NAMES = {"IW1", "IW2", "IW3"}
NISAR_DIRS = sorted(d for d in TILES_DIR.iterdir() if d.is_dir() and d.name not in _SWATH_NAMES)
print(f"S1 swaths: {S1_SWATHS}")
print(f"NISAR product dirs: {[d.name for d in NISAR_DIRS]}")

S1 swaths: ('IW1',)
NISAR product dirs: ['NISAR_L2_PR_GSLC_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001']


## 1 — Load the grid

In [2]:
def load_grid_in_bbox(grid_path, minx, miny, maxx, maxy, buffer=0.5) -> dict[str, tuple[float, float]]:
    """Stream-parse grid GeoJSON and return {name: (lon, lat)} for points within the bbox."""
    points = {}
    with open(grid_path) as fh:
        for line in fh:
            line = line.strip().rstrip(",")
            if not line.startswith('{ "type": "Feature"'):
                continue
            try:
                feat = json.loads(line)
            except json.JSONDecodeError:
                continue
            lon, lat = feat["geometry"]["coordinates"]
            if (minx - buffer) <= lon <= (maxx + buffer) and (miny - buffer) <= lat <= (maxy + buffer):
                points[feat["properties"]["name"]] = (lon, lat)
    return points


_nav = GridNavigator()


def build_grid_cells(points: dict[str, tuple[float, float]]) -> list[tuple[str, Polygon]]:
    """Build WGS84 axis-aligned rectangle cells from adjacent grid points.

    Matches grid.py get_bounded_footprint: TL/TR use BL's/BR's longitude combined
    with the next row's latitude — NOT the actual adjacent grid point coordinates,
    which shift longitude between rows (MajorTOM has independent column spacing per row).
    """
    cells = []
    for name, (lon_bl, lat_bl) in points.items():
        row, col = name.split("_")
        tl_name = _nav.move_up(row, col)
        br_name = _nav.move_right(row, col)
        if tl_name not in points or br_name not in points:
            continue
        _, lat_top = points[tl_name]   # latitude only — same lon as BL
        lon_br, _  = points[br_name]   # longitude only — same lat as BL
        cells.append((name, Polygon([
            (lon_bl, lat_bl), (lon_bl, lat_top),
            (lon_br, lat_top), (lon_br, lat_bl),
        ])))
    return cells

## 2 — Inspect the first .h5 tile structure
Run this cell once tiles are available to understand what metadata keys are present.

In [3]:
s1_h5_files = sorted(
    h5 for swath in S1_SWATHS for h5 in (TILES_DIR / swath).rglob("*.h5")
)
nisar_h5_files = sorted(h5 for d in NISAR_DIRS for h5 in d.glob("*.h5"))
print(f"Found {len(s1_h5_files)} S1 tile(s) across {S1_SWATHS}")
print(f"Found {len(nisar_h5_files)} NISAR tile(s)")

# Quick inspection of first tile of each type
for label, files in [("S1", s1_h5_files), ("NISAR", nisar_h5_files)]:
    if not files:
        continue
    with h5py.File(files[0], "r") as f:
        if "metadata/Abstracted_Metadata" in f:
            a = dict(f["metadata/Abstracted_Metadata"].attrs)
            print(f"\n{label} ({files[0].name}) — Abstracted_Metadata keys: {sorted(a.keys())}")

Found 122 S1 tile(s) across ('IW1',)
Found 389 NISAR tile(s)

S1 (126U_440R.h5) — Abstracted_Metadata keys: ['ABS_ORBIT', 'ACQUISITION_MODE', 'BEAMS', 'DEM', 'MISSION', 'PASS', 'PROC_TIME', 'PRODUCT', 'PRODUCT_TYPE', 'Processing_system_identifier', 'REL_ORBIT', 'SAMPLE_TYPE', 'SPH_DESCRIPTOR', 'STATE_VECTOR_TIME', 'SWATH', 'VECTOR_SOURCE', 'abs_calibration_flag', 'algorithm', 'ant_elev_corr_flag', 'antenna_pointing', 'avg_scene_height', 'azimuth_bandwidth', 'azimuth_looks', 'azimuth_spacing', 'bistatic_correction_applied', 'calibration_factor', 'centre_heading', 'centre_heading2', 'centre_lat', 'centre_lon', 'chirp_power', 'coregistered_stack', 'data_take_id', 'external_calibration_file', 'firstValidLineTime', 'firstValidPixel', 'first_far_lat', 'first_far_long', 'first_line_time', 'first_near_lat', 'first_near_long', 'geo_ref_system', 'inc_angle_comp_flag', 'incidence_far', 'incidence_near', 'is_terrain_corrected', 'lastValidLineTime', 'lastValidPixel', 'last_far_lat', 'last_far_long'

## 2.5 — Dimensions & metadata completeness
Full HDF5 structure, array shapes, and a side-by-side metadata comparison for one tile of each sensor type.

In [4]:
# HDF5 structure walker
def walk_h5(path):
    items = []
    def _visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            items.append((name, 'dataset', obj.shape, str(obj.dtype)))
        elif isinstance(obj, h5py.Group):
            items.append((name, 'group', len(dict(obj.attrs)), ''))
    with h5py.File(path, 'r') as f:
        f.visititems(_visitor)
        root_attrs = dict(f.attrs)
    return items, root_attrs


def print_h5_structure(path, label):
    items, root_attrs = walk_h5(path)
    print(f"\n{'='*60}")
    print(f"{label}: {path.name}")
    print(f"{'='*60}")
    if root_attrs:
        print(f"  /  (root attrs): {sorted(root_attrs.keys())}")
    for name, kind, *rest in items:
        if kind == 'dataset':
            shape, dtype = rest
            print(f"  /{name:<45}  {str(shape):<20} {dtype}")
        else:
            n_attrs = rest[0]
            suffix = f"  [{n_attrs} attrs]" if n_attrs else ''
            print(f"  /{name:<45}  <group>{suffix}")


def get_abstracted_metadata(path):
    with h5py.File(path, 'r') as f:
        if 'metadata/Abstracted_Metadata' in f:
            return dict(f['metadata/Abstracted_Metadata'].attrs)
    return {}


def get_top_metadata(path):
    with h5py.File(path, 'r') as f:
        if 'metadata' in f:
            return dict(f['metadata'].attrs)
    return {}


# One sample per sensor type
samples = []
if s1_h5_files:
    samples.append(('S1', s1_h5_files[0]))
if nisar_h5_files:
    samples.append(('NISAR', nisar_h5_files[0]))

for label, path in samples:
    print_h5_structure(path, label)


# Side-by-side Abstracted_Metadata comparison
if len(samples) == 2:
    s1_abst    = get_abstracted_metadata(samples[0][1])
    nisar_abst = get_abstracted_metadata(samples[1][1])
    nisar_top  = get_top_metadata(samples[1][1])

    all_keys = sorted(set(s1_abst) | set(nisar_abst))
    print(f"\n{'='*60}")
    print('Abstracted_Metadata field comparison (S1 vs NISAR)')
    print(f"{'='*60}")
    print(f"  {'Field':<45}  {'S1 value':^22}  {'NISAR value':^22}")
    print(f"  {'-'*45}  {'-'*22}  {'-'*22}")
    for k in all_keys:
        s1_val    = repr(s1_abst[k])[:20]    if k in s1_abst    else '—'
        nisar_val = repr(nisar_abst[k])[:20] if k in nisar_abst else 'MISSING'
        flag = ' <--' if nisar_val == 'MISSING' else ''
        print(f"  {k:<45}  {s1_val:<22}  {nisar_val:<22}{flag}")

    missing = [k for k in s1_abst if k not in nisar_abst]
    print(f"\nS1 fields absent in NISAR: {len(missing)} / {len(s1_abst)}")
    for k in missing:
        print(f"  - {k}")

    print(f"\nNISAR /metadata group attrs (spatial info):")
    for k, v in sorted(nisar_top.items()):
        print(f"  {k:<30}: {v!r}")



S1: 126U_440R.h5
  /bands                                          <group>
  /bands/Alpha                                    (1000, 1000)         float32
  /bands/Anisotropy                               (1000, 1000)         float32
  /bands/Entropy                                  (1000, 1000)         float32
  /bands/elevation                                (1000, 1000)         float32
  /bands/localIncidenceAngle                      (1000, 1000)         float32
  /metadata                                       <group>
  /metadata/Abstracted_Metadata                   <group>  [90 attrs]
  /metadata/Abstracted_Metadata/BurstBoundary     <group>
  /metadata/Abstracted_Metadata/BurstBoundary/IW1  <group>  [1 attrs]
  /metadata/Abstracted_Metadata/BurstBoundary/IW1/Burst0  <group>  [8 attrs]
  /metadata/Abstracted_Metadata/BurstBoundary/IW1/Burst0/FirstLineBoundaryPoints  <group>
  /metadata/Abstracted_Metadata/BurstBoundary/IW1/Burst0/FirstLineBoundaryPoints/BoundaryPoint.1  <group> 

## 3 — Extract tile footprints
Reads spatial bounds from each .h5 file and builds WGS84 polygons.
Supports both NISAR-style metadata (`x_min/y_min/x_max/y_max/epsg`) and
SNAP/Sentinel-1 corner-coordinate metadata (`first_near_lat/lon`, etc.).

In [5]:
def corners_from_h5(path: Path) -> Polygon:
    """Return the actual tile footprint as a 4-corner polygon in WGS84.
    For S1: corners from Abstracted_Metadata in SAR geometry order (BL→TL→TR→BR).
    For NISAR: UTM bbox corners reprojected to WGS84.
    """
    with h5py.File(path, "r") as f:
        if "metadata/Abstracted_Metadata" in f:
            a = dict(f["metadata/Abstracted_Metadata"].attrs)
            keys = {"first_near_lat", "first_near_long", "first_far_lat", "first_far_long",
                    "last_near_lat",  "last_near_long",  "last_far_lat",  "last_far_long"}
            if keys.issubset(a):
                # last_near=BL, first_near=TL, first_far=TR, last_far=BR
                return Polygon([
                    (float(a["last_near_long"]),  float(a["last_near_lat"])),
                    (float(a["first_near_long"]), float(a["first_near_lat"])),
                    (float(a["first_far_long"]),  float(a["first_far_lat"])),
                    (float(a["last_far_long"]),   float(a["last_far_lat"])),
                ])
        if "metadata" in f:
            md = dict(f["metadata"].attrs)
            if all(k in md for k in ("x_min", "y_min", "x_max", "y_max", "epsg")):
                epsg = int(md["epsg"])
                x0, y0 = float(md["x_min"]), float(md["y_min"])
                x1, y1 = float(md["x_max"]), float(md["y_max"])
                if epsg != 4326:
                    t = pyproj.Transformer.from_crs(epsg, 4326, always_xy=True)
                    corners = [t.transform(x, y) for x, y in [(x0,y0),(x0,y1),(x1,y1),(x1,y0)]]
                    return Polygon(corners)
                return Polygon([(x0,y0),(x0,y1),(x1,y1),(x1,y0)])
    raise KeyError(f"No recognised spatial metadata in {path.name}")


records = []

# S1 tiles: path structure TILES_DIR / swath / <product> / <tile>.h5
for h5 in s1_h5_files:
    try:
        records.append({"name": h5.stem, "swath": h5.parts[-3], "path": str(h5),
                         "geometry": corners_from_h5(h5)})
    except KeyError as e:
        print(f"WARN: {e}")

# NISAR tiles: path structure TILES_DIR / <product> / <tile>.h5
for h5 in nisar_h5_files:
    try:
        records.append({"name": h5.stem, "swath": "NISAR", "path": str(h5),
                         "geometry": corners_from_h5(h5)})
    except KeyError as e:
        print(f"WARN: {e}")

tile_footprints = (
    gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")
    if records
    else gpd.GeoDataFrame()
)
print(f"{len(tile_footprints)} tile footprint(s) built  "
      f"(S1: {(tile_footprints['swath'] != 'NISAR').sum() if not tile_footprints.empty else 0}, "
      f"NISAR: {(tile_footprints['swath'] == 'NISAR').sum() if not tile_footprints.empty else 0})")

# Initialise as empty; populated below if tiles exist
grid = gpd.GeoDataFrame()
grid_cells = gpd.GeoDataFrame()
grid_pts = {}

if not tile_footprints.empty:
    minx, miny, maxx, maxy = tile_footprints.total_bounds
    grid_pts = load_grid_in_bbox(GRID_PATH, minx, miny, maxx, maxy)
    print(f"{len(grid_pts):,} grid points loaded in tile extent")
    grid = gpd.GeoDataFrame(
        [{"name": n, "geometry": Point(lon, lat)} for n, (lon, lat) in grid_pts.items()],
        geometry="geometry", crs="EPSG:4326",
    )
    tile_names = set(tile_footprints["name"])
    all_cells = build_grid_cells(grid_pts)
    covered_cells = [(n, p) for n, p in all_cells if n in tile_names]
    grid_cells = gpd.GeoDataFrame(
        [{"name": n, "geometry": p} for n, p in covered_cells],
        geometry="geometry", crs="EPSG:4326",
    )
    print(f"{len(grid_cells)} grid cells built ({len(covered_cells)} covered by tiles)")

if not tile_footprints.empty:
    tile_footprints[["name", "swath", "geometry"]]
else:
    print("No tiles found — check TILES_DIR, S1_SWATHS, and NISAR_DIRS.")

511 tile footprint(s) built  (S1: 122, NISAR: 389)


1,306 grid points loaded in tile extent
462 grid cells built (462 covered by tiles)


## 3.5 — Spatial metadata: one tile & one grid cell
Shows the raw metadata driving each geometry on the map.

In [6]:
# --- Diagnostic: one S1 tile ---
sample_h5 = s1_h5_files[0]
print(f"=== S1 tile: {sample_h5.name} (swath {sample_h5.parts[-3]}) ===")
with h5py.File(sample_h5, "r") as f:
    am = dict(f["metadata/Abstracted_Metadata"].attrs)

spatial_keys = sorted(k for k in am if any(x in k.lower() for x in
    ("lat", "lon", "long", "incid", "pixel", "line", "azimuth", "range", "height", "width")))
print("Abstracted_Metadata spatial keys:")
for k in spatial_keys:
    print(f"  {k}: {am[k]}")

geom = corners_from_h5(sample_h5)
print("\nDerived polygon corners (WGS84, lon/lat):")
for i, coord in enumerate(geom.exterior.coords[:-1]):
    labels = ["BL", "TL", "TR", "BR"]
    print(f"  {labels[i]}  lon={coord[0]:.6f}  lat={coord[1]:.6f}")

# --- Diagnostic: the matching grid cell ---
tile_name = sample_h5.stem
print(f"\n=== Grid cell: {tile_name} ===")
row, col = tile_name.split("_")
tl_name = _nav.move_up(row, col)
br_name = _nav.move_right(row, col)
tr_name = _nav.move_up_right(row, col)
neighbours = {"BL (origin)": tile_name, "TL (_move_up)": tl_name,
              "BR (_move_right)": br_name, "TR": tr_name}
print("Grid corner points (WGS84, lon/lat):")
for label, name in neighbours.items():
    if name in grid_pts:
        lon, lat = grid_pts[name]
        print(f"  {label:20s}  {name:20s}  lon={lon:.6f}  lat={lat:.6f}")
    else:
        print(f"  {label:20s}  {name:20s}  NOT IN LOADED GRID")

# Print GeoJSON properties for the origin point
print("\nGeoJSON properties for origin point:")
with open(GRID_PATH) as fh:
    for line in fh:
        line = line.strip().rstrip(",")
        if not line.startswith('{ "type": "Feature"'):
            continue
        try:
            feat = json.loads(line)
        except json.JSONDecodeError:
            continue
        if feat["properties"]["name"] == tile_name:
            for k, v in feat["properties"].items():
                print(f"  {k}: {v}")
            print(f"  coordinates (lon, lat): {feat['geometry']['coordinates']}")
            break


=== S1 tile: 126U_440R.h5 (swath IW1) ===
Abstracted_Metadata spatial keys:
  avg_scene_height: 480.20013692161615
  azimuth_bandwidth: 327.0
  azimuth_looks: 1.0
  azimuth_spacing: 10.0
  centre_lat: 11.362365925549966
  centre_lon: 40.35136245814298
  firstValidLineTime: 823101968.9294221
  firstValidPixel: 0
  first_far_lat: 11.407357606986187
  first_far_long: 40.39739583079535
  first_line_time: b'30-JAN-2026 15:26:31.801042'
  first_near_lat: 11.407779521595725
  first_near_long: 40.30575492652231
  incidence_far: 36.68159513476333
  incidence_near: 30.955264279404663
  lastValidLineTime: 823101993.9743202
  lastValidPixel: 20423
  last_far_lat: 11.316946784331295
  last_far_long: 40.39695556380176
  last_line_time: b'30-JAN-2026 15:26:33.154425'
  last_near_lat: 11.31736526946108
  last_near_long: 40.30534351145039
  lat_pixel_res: 8.983152841195215e-05
  line_time_interval: 0.002055556299999998
  lon_pixel_res: 8.983152841195215e-05
  num_output_lines: 1000
  num_samples_per_li

  name: 126U_440R
  row: 126U
  col: 440R
  row_idx: 1072
  col_idx: 2405
  utm_zone: 32637
  epsg: EPSG:32637
  coordinates (lon, lat): [40.30534351145039, 11.317365269461078]


## 3.6 — Single tile alignment deep dive

### Pipeline geometry: from S1 acquisition to grid-aligned tile

**Step 1 — MajorTOM grid (WGS84, EPSG:4326)**
The grid is a global set of points in WGS84 (lon/lat), spaced ~10 km apart.
Each point has a `name` (e.g. `0025_00182`) and an `epsg` property indicating
the local UTM zone. Grid cells are axis-aligned rectangles defined by four
adjacent grid points, constructed in UTM coordinates.

**Step 2 — S1 SLC acquisition (SAR geometry)**
The raw Sentinel-1 product has a parallelogram-shaped footprint in slant-range
geometry (not map-projected). SNAP stores corner coordinates in
`Abstracted_Metadata` (`first_near_lat/long`, `last_far_lat/long`, etc.).
At this stage these corners reflect the original SAR look geometry.

**Step 3 — SNAP processing chain**
`Orbit → Calibration → DerampDemod → Deburst → TerrainCorrection (10 m)`
After terrain correction, the raster is reprojected into UTM (the local EPSG
from the grid). The output is now rectangular in UTM space.

**Step 4 — Tiling (pixel subset per grid cell)**
The pipeline computes each grid cell's UTM bounding box (`grid_cell_utm_bbox`),
converts it to a pixel region, and runs SNAP `Subset` to cut a ~1000×1000 px
tile. Each tile is saved as an `.h5` file.

**Step 5 — Metadata update (`_update_h5_corners`)**
After subsetting, the pipeline **overwrites** the original SAR corner coordinates
in `Abstracted_Metadata` with the UTM bbox corners reprojected back to WGS84.
This means the `.h5` metadata now stores the actual tile extent (= the grid
cell extent), not the original acquisition geometry.

**Step 6 — This notebook reads both sources and compares them**
- **Blue tile polygon (pipeline output):** read directly from the `.h5` file's
  `Abstracted_Metadata` corner attributes (WGS84, written by step 5).
  This reflects what the pipeline actually produced.
- **Red grid cell polygon (ground truth):** reconstructed independently from
  the MajorTOM grid GeoJSON points (WGS84). No pipeline output is used —
  it comes straight from the grid source file (`grid_10km.geojson`).

The fact that both polygons overlap exactly confirms that the pipeline is
cutting tiles to the correct grid cell extents. They are both rectangular
because the grid cells are defined as axis-aligned rectangles in UTM.

In [7]:
if not s1_h5_files:
    print("No tiles found — run cells 1–3 first.")
elif not grid_pts:
    print("Grid not loaded yet — run cell 3 first.")
else:
    sample_h5 = s1_h5_files[0]
    tile_name = sample_h5.stem
    row, col = tile_name.split("_")

    # Raster shape
    with h5py.File(sample_h5) as f:
        bands = list(f.get("bands", {}).keys())
        shape = f[f"bands/{bands[0]}"].shape if bands else "no bands found"
    print(f"Tile: {tile_name}   raster shape: {shape}")

    # Tile corners from h5 metadata
    geom_tile = corners_from_h5(sample_h5)
    tile_coords = dict(zip(["BL", "TL", "TR", "BR"], geom_tile.exterior.coords[:4]))

    # Correct grid cell corners: WGS84 axis-aligned rectangle
    tl_name = _nav.move_up(row, col)
    br_name = _nav.move_right(row, col)
    lon_bl, lat_bl = grid_pts.get(tile_name, (None, None))
    lat_top = grid_pts[tl_name][1] if tl_name in grid_pts else None
    lon_br  = grid_pts[br_name][0] if br_name in grid_pts else None
    grid_corners = {
        "BL": (lon_bl,  lat_bl),
        "TL": (lon_bl,  lat_top),
        "TR": (lon_br,  lat_top),
        "BR": (lon_br,  lat_bl),
    }

    # Per-corner delta in UTM metres
    t_utm = pyproj.Transformer.from_crs(4326, 32637, always_xy=True)
    print(f"\n{'Corner':<6}  {'Tile lon':>12} {'Tile lat':>12}  {'Grid lon':>12} {'Grid lat':>12}  {'Δ (m)':>8}")
    for label in ["BL", "TL", "TR", "BR"]:
        tc = tile_coords[label]
        gc = grid_corners[label]
        if None in gc:
            print(f"  {label:<6}  (grid corner not in loaded extent)")
            continue
        tx, ty = t_utm.transform(*tc)
        gx, gy = t_utm.transform(*gc)
        delta = ((tx - gx)**2 + (ty - gy)**2)**0.5
        print(f"  {label:<6}  {tc[0]:>12.6f} {tc[1]:>12.6f}  {gc[0]:>12.6f} {gc[1]:>12.6f}  {delta:>8.1f}")

    # Zoomed folium map: tile (blue filled) vs grid cell (red dashed)
    if lon_bl and lat_bl and lat_top and lon_br:
        cx = (lon_bl + lon_br) / 2
        cy = (lat_bl + lat_top) / 2
        m2 = folium.Map(location=[cy, cx], zoom_start=13, tiles="OpenStreetMap")
        folium.GeoJson(
            geom_tile.__geo_interface__,
            style_function=lambda _: {"color": "#1f77b4", "fillColor": "#1f77b4",
                                       "fillOpacity": 0.3, "weight": 2},
            tooltip=f"SAR tile: {tile_name}",
        ).add_to(m2)
        cell_poly = Polygon([(lon_bl, lat_bl), (lon_bl, lat_top),
                              (lon_br, lat_top), (lon_br, lat_bl)])
        folium.GeoJson(
            cell_poly.__geo_interface__,
            style_function=lambda _: {"color": "red", "fillOpacity": 0,
                                       "weight": 2.5, "dashArray": "6 4"},
            tooltip=f"Grid cell: {tile_name}",
        ).add_to(m2)
        display(m2)


Tile: 126U_440R   raster shape: (1000, 1000)

Corner      Tile lon     Tile lat      Grid lon     Grid lat     Δ (m)
  BL         40.305344    11.317365     40.305344    11.317365       0.0
  TL         40.305755    11.407780     40.305344    11.407186      79.6
  TR         40.397396    11.407358     40.396947    11.407186      52.6
  BR         40.396956    11.316947     40.396947    11.317365      46.3


## 4 — Interactive map: grid + tiles on basemap

In [8]:
try:
    _tf = tile_footprints
except NameError:
    _tf = gpd.GeoDataFrame()

if _tf.empty:
    print("No tiles to display yet — run cells 1–3 first.")
else:
    cx = _tf.geometry.to_crs("EPSG:3857").centroid.to_crs("EPSG:4326").x.mean()
    cy = _tf.geometry.to_crs("EPSG:3857").centroid.to_crs("EPSG:4326").y.mean()
    m = folium.Map(location=[cy, cx], zoom_start=8, tiles="OpenStreetMap")

    SWATH_COLORS = {"IW1": "#1f77b4", "IW2": "#ff7f0e", "IW3": "#2ca02c", "NISAR": "#d62728"}

    # Layer 1: actual SAR tile footprints
    for swath, color in SWATH_COLORS.items():
        subset = _tf[_tf["swath"] == swath]
        if subset.empty:
            continue
        layer = folium.FeatureGroup(name=f"SAR footprint – {swath}", show=True)
        for _, row in subset.iterrows():
            folium.GeoJson(
                row["geometry"].__geo_interface__,
                style_function=lambda _, c=color: {
                    "color": c, "fillColor": c, "fillOpacity": 0.35, "weight": 1.5,
                },
                tooltip=row["name"],
            ).add_to(layer)
        layer.add_to(m)

    # Layer 2: grid cells (dashed outlines, no fill)
    if not grid_cells.empty:
        for swath, color in SWATH_COLORS.items():
            swath_names = set(_tf[_tf["swath"] == swath]["name"])
            subset = grid_cells[grid_cells["name"].isin(swath_names)]
            if subset.empty:
                continue
            layer = folium.FeatureGroup(name=f"Grid cell – {swath}", show=True)
            folium.GeoJson(
                subset.__geo_interface__,
                style_function=lambda _, c=color: {
                    "color": c, "fillColor": c, "fillOpacity": 0.0,
                    "weight": 2.5, "dashArray": "5 4",
                },
                tooltip=folium.GeoJsonTooltip(fields=["name"]),
            ).add_to(layer)
            layer.add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save("/tmp/grid_tiles_map.html")
    display(m)

## 5 — Alignment assertion
Each tile footprint should contain exactly one grid point (its defining center).

In [9]:
if tile_footprints.empty:
    print("No tiles to check.")
else:
    tile_names = set(tile_footprints["name"])
    matched = tile_names & set(grid_pts.keys())
    unmatched = tile_names - matched
    print(f"Tiles with matching grid point: {len(matched)} / {len(tile_footprints)}")
    if unmatched:
        print(f"WARN: {len(unmatched)} tile(s) have no matching grid point name: {sorted(unmatched)}")
    else:
        print("All tile names have a corresponding grid point. Alignment OK.")

    # --- S1 vs NISAR coverage comparison ---
    s1_names  = set(tile_footprints[tile_footprints["swath"] != "NISAR"]["name"])
    nisar_names = set(tile_footprints[tile_footprints["swath"] == "NISAR"]["name"])
    shared     = s1_names & nisar_names
    s1_only    = s1_names - nisar_names
    nisar_only = nisar_names - s1_names

    print(f"\nS1 tiles:          {len(s1_names)}")
    print(f"NISAR tiles:       {len(nisar_names)}")
    print(f"Shared grid cells: {len(shared)}")
    print(f"S1 only:           {len(s1_only)}")
    print(f"NISAR only:        {len(nisar_only)}")
    if s1_only:
        print(f"  S1-only cells:    {sorted(s1_only)}")
    if nisar_only:
        print(f"  NISAR-only cells: {sorted(nisar_only)}")

Tiles with matching grid point: 462 / 511
All tile names have a corresponding grid point. Alignment OK.

S1 tiles:          122
NISAR tiles:       389
Shared grid cells: 49
S1 only:           73
NISAR only:        340
  S1-only cells:    ['126U_440R', '126U_441R', '126U_442R', '126U_443R', '126U_444R', '127U_440R', '127U_441R', '127U_442R', '127U_443R', '127U_444R', '127U_445R', '127U_446R', '127U_447R', '127U_448R', '128U_440R', '128U_441R', '128U_442R', '128U_443R', '128U_444R', '128U_445R', '128U_446R', '128U_447R', '128U_448R', '129U_439R', '129U_440R', '129U_441R', '129U_442R', '129U_443R', '130U_439R', '130U_440R', '130U_441R', '130U_442R', '130U_443R', '131U_439R', '131U_440R', '131U_441R', '131U_442R', '131U_443R', '132U_438R', '132U_439R', '132U_440R', '132U_441R', '132U_442R', '133U_438R', '133U_439R', '133U_440R', '133U_441R', '133U_442R', '134U_437R', '134U_438R', '134U_439R', '134U_440R', '134U_441R', '135U_437R', '135U_438R', '135U_439R', '135U_440R', '135U_441R', '136U_4

In [10]:
# --- Per-cell boundary alignment: S1 vs NISAR vs grid cell ---
# For each shared grid cell, compare all three footprints corner-by-corner.

if not shared:
    print("No shared grid cells between S1 and NISAR — nothing to compare.")
else:
    t_utm = pyproj.Transformer.from_crs(4326, 32637, always_xy=True)
    s1_fp    = tile_footprints[tile_footprints["swath"] != "NISAR"].set_index("name")
    nisar_fp = tile_footprints[tile_footprints["swath"] == "NISAR"].set_index("name")

    mismatches = []
    for cell in sorted(shared):
        geom_s1    = s1_fp.loc[cell, "geometry"]
        geom_nisar = nisar_fp.loc[cell, "geometry"]

        # Reproject both to UTM for distance in metres
        def to_utm_coords(geom):
            return [t_utm.transform(lon, lat) for lon, lat in geom.exterior.coords[:-1]]

        s1_utm    = to_utm_coords(geom_s1)
        nisar_utm = to_utm_coords(geom_nisar)

        if len(s1_utm) != len(nisar_utm):
            mismatches.append((cell, "different vertex count"))
            continue

        max_delta = max(
            ((sx-nx)**2 + (sy-ny)**2)**0.5
            for (sx,sy),(nx,ny) in zip(s1_utm, nisar_utm)
        )
        if max_delta > 1.0:  # > 1 m mismatch
            mismatches.append((cell, f"max corner delta = {max_delta:.1f} m"))

    if not mismatches:
        print(f"All {len(shared)} shared cells: S1 and NISAR footprints agree within 1 m.")
    else:
        print(f"{len(mismatches)} / {len(shared)} cells have mismatches:")
        for cell, reason in mismatches:
            print(f"  {cell}: {reason}")

    # Zoomed map of a shared cell: S1 (blue), NISAR (red), grid cell (dashed black)
    sample_cell = sorted(shared)[0]
    geom_s1    = s1_fp.loc[sample_cell, "geometry"]
    geom_nisar = nisar_fp.loc[sample_cell, "geometry"]
    row_c, col_c = sample_cell.split("_")
    tl_n = _nav.move_up(row_c, col_c)
    br_n = _nav.move_right(row_c, col_c)
    if sample_cell in grid_pts and tl_n in grid_pts and br_n in grid_pts:
        lon_bl, lat_bl = grid_pts[sample_cell]
        lat_top = grid_pts[tl_n][1]
        lon_br  = grid_pts[br_n][0]
        grid_cell_poly = Polygon([(lon_bl,lat_bl),(lon_bl,lat_top),(lon_br,lat_top),(lon_br,lat_bl)])
        cx = (lon_bl + lon_br) / 2
        cy = (lat_bl + lat_top) / 2
        m3 = folium.Map(location=[cy, cx], zoom_start=13, tiles="OpenStreetMap")
        for geom, color, label in [
            (geom_s1,    "#1f77b4", f"S1 – {sample_cell}"),
            (geom_nisar, "#d62728", f"NISAR – {sample_cell}"),
            (grid_cell_poly, "black", f"Grid cell – {sample_cell}"),
        ]:
            folium.GeoJson(
                geom.__geo_interface__,
                style_function=lambda _, c=color: {
                    "color": c,
                    "fillOpacity": 0.15 if c != "black" else 0,
                    "weight": 2.5,
                    "dashArray": "6 4" if c == "black" else None,
                },
                tooltip=label,
            ).add_to(m3)
        display(m3)

49 / 49 cells have mismatches:
  129U_444R: max corner delta = 8.3 m
  129U_445R: max corner delta = 8.7 m
  129U_446R: max corner delta = 10.4 m
  129U_447R: max corner delta = 9.2 m
  130U_444R: max corner delta = 8.3 m
  130U_445R: max corner delta = 9.7 m
  130U_446R: max corner delta = 10.4 m
  130U_447R: max corner delta = 10.0 m
  131U_444R: max corner delta = 9.2 m
  131U_445R: max corner delta = 9.3 m
  131U_446R: max corner delta = 9.7 m
  131U_447R: max corner delta = 10.6 m
  132U_443R: max corner delta = 9.7 m
  132U_444R: max corner delta = 9.1 m
  132U_445R: max corner delta = 8.2 m
  132U_446R: max corner delta = 8.4 m
  132U_447R: max corner delta = 8.3 m
  133U_443R: max corner delta = 9.0 m
  133U_444R: max corner delta = 10.6 m
  133U_445R: max corner delta = 10.4 m
  133U_446R: max corner delta = 8.5 m
  133U_447R: max corner delta = 6.5 m
  134U_442R: max corner delta = 9.3 m
  134U_443R: max corner delta = 9.2 m
  134U_444R: max corner delta = 9.0 m
  134U_445R: 